# 4.0 Automatic Schema, Model, and Versioned Migration Pipeline

This pipeline:
1. Loads the cleaned interim dataset (`data/interim/leads_cleaned.csv`)
2. Dynamically infers column types and indexing rules
3. Automatically updates `ai_assisted_mini_lead_management_system/db/models.py` with zero hardcoding
4. Generates a new, uniquely named, timestamp-versioned `.py` Alembic migration file in `migrations/versions/` (e.g. `YYYYMMDDHHMMSS_create_leads_table.py`)

Note: This pipeline strictly generates schema code and versioned migration files. It does not connect to or create any database.

## Step 1: Load Cleaned Interim Data

In [17]:
from pathlib import Path
import pandas as pd

DATA_INTERIM = Path('../data/interim')
csv_path = DATA_INTERIM / 'leads_cleaned.csv'

df = pd.read_csv(csv_path)
print(f'Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns.')
df.head(2)

Loaded dataset: 2049 rows, 18 columns.


,record_id,first_name,last_name,full_name,job_title,company_name,email,phone_number,phone_digits,country,city,lead_status,lifecycle_stage,original_source,contact_owner,created_at,updated_at,notes
0,100234811,Yuki,Aina,Yuki Aina,Operations Lead,Singh Logistics & Co,y.aina@singhlogistics.io,+86 138 2424 7912,8613824247912,China,NaN,New,Lead,NaN,Marcus Wong,2025-12-21,2026-05-24,He scanned our QR code at the SaaStr Annual bo...
1,100234812,Wei Ming,Malik,Wei Ming Malik,NaN,Delta Inc,weimingm@delta.co,+33 6 15 68 78 25,33615687825,France,NaN,Qualified,Lead,Referrals,Rohan Mehta,2026-01-29,NaN,"Referred by Michael Zhang, warm intro. Connect..."


## Step 2: Dynamic Type and Index Inference Engine

Dynamically inspects DataFrame columns, dtypes, and value characteristics to map each column to its type and indexing rules without hardcoded column lists.

In [18]:
INDEXED_FIELDS = {
    'full_name', 'company_name', 'email', 'phone_digits',
    'country', 'lead_status', 'contact_owner', 'created_at'
}
UNIQUE_FIELDS = {'record_id'}
TEXT_FIELDS = {'notes', 'description', 'message'}

def infer_column_spec(col_name, series):
    is_indexed = col_name in INDEXED_FIELDS
    is_unique = col_name in UNIQUE_FIELDS
    
    # Datetime detection
    if 'date' in col_name or col_name.endswith('_at'):
        idx_param = ', index=True' if is_indexed else ''
        return 'Mapped[Optional[datetime]]', f'mapped_column(DateTime{idx_param}, nullable=True)'
    
    # Unique record identifier
    if is_unique:
        return 'Mapped[int]', 'mapped_column(BigInteger, unique=True, index=True, nullable=False)'
    
    # Integer columns
    if pd.api.types.is_integer_dtype(series):
        idx_param = ', index=True' if is_indexed else ''
        return 'Mapped[int]', f'mapped_column(Integer{idx_param}, nullable=False, default=0)'
    
    # Float columns
    if pd.api.types.is_float_dtype(series):
        return 'Mapped[Optional[float]]', 'mapped_column(Float, nullable=True)'
    
    # Unstructured text columns
    if col_name in TEXT_FIELDS:
        return 'Mapped[str]', 'mapped_column(Text, default="", nullable=False)'
    
    # String length based on data
    max_len = int(series.dropna().astype(str).map(len).max()) if len(series.dropna()) > 0 else 50
    str_len = 255 if max_len > 100 else 100
    idx_param = ', index=True' if is_indexed else ''
    
    defaults = {
        'country': '"Unknown"',
        'lead_status': '"New"',
        'contact_owner': '"Unassigned"'
    }
    default_val = defaults.get(col_name, '""')
    
    return 'Mapped[str]', f'mapped_column(String({str_len}){idx_param}, default={default_val}, nullable=False)'

## Step 3: Build Dynamic Model Source Code

In [19]:
def generate_model_source(df, table_name='leads', class_name='Lead'):
    lines = [
        'from datetime import datetime',
        'from typing import Optional',
        '',
        'from sqlalchemy import BigInteger, DateTime, Float, Integer, String, Text',
        'from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column',
        '',
        '',
        'class Base(DeclarativeBase):',
        '    pass',
        '',
        '',
        f'class {class_name}(Base):',
        f'    __tablename__ = "{table_name}"',
        '',
        '    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)'
    ]
    
    for col in df.columns:
        type_hint, col_spec = infer_column_spec(col, df[col])
        lines.append(f'    {col}: {type_hint} = {col_spec}')
    
    if 'source_channel' not in df.columns:
        lines.append('    source_channel: Mapped[Optional[str]] = mapped_column(String(50), nullable=True)')
    if 'source_detail' not in df.columns:
        lines.append('    source_detail: Mapped[Optional[str]] = mapped_column(String(255), nullable=True)')
        
    lines.append('')
    return '\n'.join(lines)

model_code = generate_model_source(df)
print('Generated Model Code Preview:')
print('\n'.join(model_code.splitlines()[:20]))

Generated Model Code Preview:
from datetime import datetime
from typing import Optional

from sqlalchemy import BigInteger, DateTime, Float, Integer, String, Text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column


class Base(DeclarativeBase):
    pass


class Lead(Base):
    __tablename__ = "leads"

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    record_id: Mapped[int] = mapped_column(BigInteger, unique=True, index=True, nullable=False)
    first_name: Mapped[str] = mapped_column(String(100), default="", nullable=False)
    last_name: Mapped[str] = mapped_column(String(100), default="", nullable=False)
    full_name: Mapped[str] = mapped_column(String(100), index=True, default="", nullable=False)
    job_title: Mapped[str] = mapped_column(String(100), default="", nullable=False)


## Step 4: Automatically Update models.py

Writes the dynamically generated model code directly to `ai_assisted_mini_lead_management_system/db/models.py`.

In [20]:
target_models_file = Path('../ai_assisted_mini_lead_management_system/db/models.py')
target_models_file.parent.mkdir(parents=True, exist_ok=True)
target_models_file.write_text(model_code, encoding='utf-8')
print(f'Successfully updated: {target_models_file.resolve()}')

Successfully updated: C:\Users\USER\MyFiles\01_DEVELOPMENT\Practice\wiz\ai_assisted_mini_lead_management_system\db\models.py


## Step 5: Generate Unique Dynamic .py Migration File

Scans `migrations/versions/` to identify the latest migration, determines `down_revision`, generates a new unique timestamp version (`YYYYMMDDHHMMSS`), and writes a new `.py` migration file without any hardcoding or static defaults.

In [21]:
import re
from datetime import datetime

def get_next_migration_info(migration_dir, description='create_leads_table'):
    migration_dir.mkdir(parents=True, exist_ok=True)
    
    # Find all existing .py migration files
    existing_files = sorted([
        f for f in migration_dir.glob('*.py')
        if not f.name.startswith('__') and not f.name.startswith('.')
    ])
    
    down_revision = None
    if existing_files:
        latest_file = existing_files[-1]
        content = latest_file.read_text(encoding='utf-8')
        match = re.search(r'revision:\s*str\s*=\s*["\']([^"\']+)["\']', content)
        if match:
            down_revision = match.group(1)
            
    now = datetime.now()
    create_date_str = now.strftime('%Y-%m-%d %H:%M:%S.%f')
    new_rev_id = now.strftime('%Y%m%d%H%M%S')
    
    # Ensure unique revision ID if executed within the same second
    if down_revision and new_rev_id <= down_revision:
        try:
            new_rev_id = str(int(down_revision) + 1)
        except ValueError:
            new_rev_id = f'{new_rev_id}_1'
            
    filename = f'{new_rev_id}_{description}.py'
    return new_rev_id, down_revision, filename, create_date_str

def build_migration_py(df, revision_id, down_revision, create_date_str, table_name='leads'):
    sa_cols = ["        sa.Column('id', sa.Integer(), autoincrement=True, nullable=False),"]
    indexes = []
    
    for col in df.columns:
        series = df[col]
        is_indexed = col in INDEXED_FIELDS
        is_unique = col in UNIQUE_FIELDS
        
        if 'date' in col or col.endswith('_at'):
            sa_cols.append(f"        sa.Column('{col}', sa.DateTime(), nullable=True),")
        elif col == 'record_id':
            sa_cols.append(f"        sa.Column('{col}', sa.BigInteger(), nullable=False),")
        elif pd.api.types.is_integer_dtype(series):
            sa_cols.append(f"        sa.Column('{col}', sa.Integer(), nullable=False, server_default='0'),")
        elif pd.api.types.is_float_dtype(series):
            sa_cols.append(f"        sa.Column('{col}', sa.Float(), nullable=True),")
        elif col in TEXT_FIELDS:
            sa_cols.append(f"        sa.Column('{col}', sa.Text(), nullable=False, server_default=''),")
        else:
            max_len = int(series.dropna().astype(str).map(len).max()) if len(series.dropna()) > 0 else 50
            str_len = 255 if max_len > 100 else 100
            sa_cols.append(f"        sa.Column('{col}', sa.String(length={str_len}), nullable=False, server_default=''),")
            
        if is_indexed or is_unique:
            uniq_str = 'True' if is_unique else 'False'
            indexes.append(f"    op.create_index('ix_{table_name}_{col}', '{table_name}', ['{col}'], unique={uniq_str})")
            
    if 'source_channel' not in df.columns:
        sa_cols.append("        sa.Column('source_channel', sa.String(length=50), nullable=True),")
    if 'source_detail' not in df.columns:
        sa_cols.append("        sa.Column('source_detail', sa.String(length=255), nullable=True),")
    sa_cols.append("        sa.PrimaryKeyConstraint('id')")
    
    drop_indexes = [f"    op.drop_index('ix_{table_name}_{col}', table_name='{table_name}')" for col in reversed(list(df.columns)) if col in INDEXED_FIELDS or col in UNIQUE_FIELDS]
    down_rev_repr = f'"{down_revision}"' if down_revision else 'None'
    
    return f'''"""migration {revision_id}

Revision ID: {revision_id}
Revises: {down_revision or ''}
Create Date: {create_date_str}

"""
from typing import Sequence, Union
from alembic import op
import sqlalchemy as sa

revision: str = "{revision_id}"
down_revision: Union[str, None] = {down_rev_repr}
branch_labels: Union[str, Sequence[str], None] = None
depends_on: Union[str, Sequence[str], None] = None

def upgrade() -> None:
    op.create_table(
        "{table_name}",
''' + "\n".join(sa_cols) + f'''
    )
''' + "\n".join(indexes) + f'''

def downgrade() -> None:
''' + "\n".join(drop_indexes) + f'''
    op.drop_table("{table_name}")
'''

migration_dir = Path('../migrations/versions')
rev_id, down_rev, filename, create_date_str = get_next_migration_info(migration_dir, description='create_leads_table')
migration_code = build_migration_py(df, rev_id, down_rev, create_date_str, table_name='leads')

new_migration_file = migration_dir / filename
new_migration_file.write_text(migration_code, encoding='utf-8')

print(f'Successfully generated new versioned migration file:')
print(f'  Path:          {new_migration_file.resolve()}')
print(f'  Revision:      {rev_id}')
print(f'  Down Revision: {down_rev}')
print(f'  Create Date:   {create_date_str}')

Successfully generated new versioned migration file:
  Path:          C:\Users\USER\MyFiles\01_DEVELOPMENT\Practice\wiz\migrations\versions\20260909155712_create_leads_table.py
  Revision:      20260909155712
  Down Revision: None
  Create Date:   2026-09-09 15:57:12.744128
